# Final Project
A narative is also available on [the dedicated GitHub page](https://lafortua.github.io/Final_Project_CS-215/index.html). It excludes the super technical parts in favor of telling a more cohesive story.

The [resources that I used are listed on that GitHub page](https://lafortua.github.io/Final_Project_CS-215/index.html#:~:text=Most%20resources%20used%20are%20linked%20when%20referenced%2E%20Expand%20to%20see%20more%20extensive%20list%20is%20below) along with explanations of what specifically I used that resource for.

In [104]:
import pandas as pd
import numpy as np
from datetime import datetime
import nltk

In [105]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Obtaining the data

In the interest of processing speed, Java was used to fetch and clean the transcript and IMDb data. There were many errors in the data such as the letter "l" (lowercase l as in "lifeguard") occasionally being in places where the letter "I" (capital i as in "I am") should be. I knew that I would need to rerun the cleaning code many times as I worked to fix the errors. In addition, working with string objects is especially computationally expensive. Thus Java's speed and controll it affords would be helpful.

The Java code is included in the GitHub repository [and can be found here](https://github.com/lafortua/Final_Project_CS-215/blob/main/Java_Code/).

The process in order is:
1.  The [getLinks.java program](https://github.com/lafortua/Final_Project_CS-215/blob/main/Java_Code/Crawl_and_Clean/getLinks.java) uses Jsoup (an API tool for Java) to prepare a list of all the links on the main [Subslikescript page for Cheers](https://subslikescript.com/series/Cheers-83399) and saves the list into a .txt file formatted like a .csv file.
2.  The [getScript.java program](https://github.com/lafortua/Final_Project_CS-215/blob/main/Java_Code/Crawl_and_Clean/getScript.java) uses the list of links produced by the getLinks.java program. It visits each page link on the list one at a time and (using Jsoup) parses the html of that page to extract the lines. The program also recombines lines that were split in half by the transcription. (This is done by detecting punctuation that would be at the end of a sentence.) Each line is then written to a .txt file with the format "Text; Season; Episode; EpisodeTitle." Java does not have a built in way to easily interact with .csv files, but .csv files are just specifically organized .txt files. Each line of dialogue is formatted using semicolons as the delimiter and just one line of dialogue is on each line of the file. [The GitHub page explains this decision regarding the delimiter.](https://lafortua.github.io/Final_Project_CS-215/index.html#:~:text=I%20made,delimiter,-%2E) (The first line of the file is often interpreted as the column name. Thus I made the first line of the text file "Text;Season;Episode;EpisodeTitle.")
3. The [cleanScript.java program](https://github.com/lafortua/Final_Project_CS-215/blob/main/Java_Code/Crawl_and_Clean/cleanScript.java) reads in the file that is created by the getScript.java program. This program then performs a lot of hand-written operations on each string of text (only the first column of the file) to manualy check for and correct common errors that I observed in the original. This program then outputs a new file that is formatted the same as the one produced by the getScript.java program, but uses the corrected text.
4.  The [IMDbScraper.java program](https://github.com/lafortua/Final_Project_CS-215/blob/main/Java_Code/Crawl_and_Clean/IMDbScraper.java) parses html web pages that I manualy downloaded from IMDb. It was not possible to access the pages directly online because the pages were pure JavaScript. The program then writes the information for each episode into a new .txt file. The file is formatted as "Season;Episode;AirDate;Rating;Votes."
5.  The [combineScriptRatings.java program](https://github.com/lafortua/Final_Project_CS-215/blob/main/Java_Code/Crawl_and_Clean/combineScriptRatings.java) reads in both the file created by the cleanScript.java program and the IMDbScraper.java program. For each line it adds the new IMDb data with the matching season and episode number. The program then write the complete transcript into a new .txt file. The file is formatted as "Text;Season;Episode;EpisodeTitle;AirDate;Rating;Votes." Since I was very careful with the formatting of the .txt file, I can convert it into a .csv file by simply renaming the file to have the .csv file extenstion instead of .txt. This is the final transcript file named "Cheers_Lines.csv" that I then load into the Google Colab notebook.

In [106]:
# Read in the Cheers_Lines.csv file, making sure to identify the separator/delimiter as ";". (The assumed default is ",".)
df = pd.read_csv('/content/drive/MyDrive/CS215-Spring-AUGUSTUS/Projects/Final_Project/Data/Cheers_Lines.csv', sep=';')

# Convert the strings in AirDate to datetime objects, making them easier to interact with.
df['AirDate'] = pd.to_datetime(df['AirDate'])

df.head()

,Text,Season,Episode,EpisodeTitle,AirDate,Rating,Votes
0,"How about a beer, chief?",1,1,Give Me a Ring Sometime,1982-09-30,8.2,1700
1,How about an ID?,1,1,Give Me a Ring Sometime,1982-09-30,8.2,1700
2,An ID?,1,1,Give Me a Ring Sometime,1982-09-30,8.2,1700
3,That's very flattering!,1,1,Give Me a Ring Sometime,1982-09-30,8.2,1700
4,Wait till I tell the missus.,1,1,Give Me a Ring Sometime,1982-09-30,8.2,1700


## Creating an index for character relevance

I explain in depth my logic for creating an index of character relevance in [Part 1 of the Data Analysis section on my GitHub page](https://lafortua.github.io/Final_Project_CS-215/index.html#:~:text=Part%201%3A%20How%20can%20we%20find%20characters%20if%20we%20don%27t%20have%20speaker%20names).

To create the index file, I again used Java because of how much string interaction it required. This Java code is also [available on the GitHub repository](https://github.com/lafortua/Final_Project_CS-215/tree/main/Java_Code/Index_Lines).

The Name.java file is the program that runs while the Transcript.java and Line.java files each define a new class. Each line from the Cheers_Lines.csv is made into a Line object. I made a lot of functions that can be called with a Line object. One of the functions being hasName(). When calling the function you pass a character's name as a variable, and it returns a boolean. (True if the character's name is spoken in the line.) It automatically checks for all different names that the character goes by. A Transcript object is essentially an array of Line objects. This makes it easier to loop through all of the lines. One of the functions that I made for the Transcript object is isNameNearLine(). When calling this function you pass a character's name as a variable, and it returns a boolean. (True if the character's name is spoken in the line or in a line "nearby" which is defined as within two lines before or after.)

Here I have the Name.java file take in the Cheers_Lines file and uses the isNameNearLine() function to construct a file that has the relevant lines marked. The file that it outputs is the [Cheers_Name_Index.csv](https://github.com/lafortua/Final_Project_CS-215/blob/main/Cheers_Name_Index.csv) file. We can load this file into this notebook too.

In [107]:
# Read in the Cheers_Name_Index.csv file, making sure to identify the separator/delimiter as ";". (The assumed default is ",".)
df_names = pd.read_csv('/content/drive/MyDrive/CS215-Spring-AUGUSTUS/Projects/Final_Project/Big_Data/Cheers_Name_Index.csv', sep=';')

df_names.head()

,Sam,Carla,Cliff,Norm,Frasier,Woody,Rebecca,Diane,Lilith,Coach,Nick,Eddie,Kelly,Gary
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Now that we know which characters are relevant in a given interaction, let's start paying attention to what is said in those interactions. Instead of simply choosing specific words to look at, let's use statistics to choose the words for us.

## Term Frequency - Inverse Document Frequency

The first method that I used to try to find words of interest is called Term Frequency - Inverse Document Frequency. Often reffered to as TF-IDF, it is a common measure used when investigating word frequency in a large dataset.

A high weighted TF-IDF is the result of a high term frequency in a given document and a low document frequency of the term in the whole collection of documents. The idea of this is to measure how important a given term is by weighting the document frequency by the specificity of the term. For example, in most documents the word "the" would have the highest document frequency as it is the most often word used in the English language. This also means that it has a low specificity, which lowers its importance.

I will use the TF-IDF vectorizor from scikit-learn to do the calculations.

In [108]:
# Import the TF-IDF vectorizor from scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

# Make sure the text passed does not have any empty parts
df = df.dropna(subset=['Text'])

# Create the vectorizer object with a minimum document frequency of 10. This means it will exclude words that are used in less than 10 documents. (In this case, each line of dialogue is a "document.")
tfidf = TfidfVectorizer(min_df=10)
result = tfidf.fit_transform(df['Text'])

In [109]:
# Print out some of the words at the top of the list.
idf_values = []
for ele1, ele2 in zip(tfidf.get_feature_names_out(), tfidf.idf_):
    idf_values.append((ele1, ele2))

# Sort the idf_values list by the IDF value in decending order
idf_values_sorted = sorted(idf_values, key=lambda x: x[1], reverse=True)

print('\nTop 5 words in idf_values_sorted:')
# Only show the first 5 words in the list
for word, idf_value in idf_values_sorted[:5]:
    print(word, ':', idf_value)


Top 5 words in idf_values_sorted:
19 : 10.225649854243818
37 : 10.225649854243818
absolute : 10.225649854243818
adults : 10.225649854243818
advertising : 10.225649854243818


Aw man! Using TF-IDF on this dataset did not give very meaningful results. There were dozens of words that were tied for the same score. This is because, even with a large dataset of 111,697 lines, there were many words that appeared the exact same number of times as each other. The following table shows the first five words in the list of all terms used. Each of these words was spoken nine times, in nine different lines throughout the whole tv show.

## Log Likelihood with SUBTLEXus

Unsatisfied with the TF-IDF results, I started to look for a method that would tell me which words are used uniquely often in the tv show. This search led me to using a Log-Likelihood test against a reference corpus. This would tell me how the word frequencies in my dataset compared to the reference. I decided to use a dataset called the [SUBTLEXus corpus](https://www.ugent.be/pp/experimentele-psychologie/en/research/documents/subtlexus) as the reference. This dataset of word frequency is based on the subtitles of English and American movies and tv shows. This makes it a more sensible choice when compared to many other datasets, such as the Brown Corpus, which are based on written documents or are far out of date.

In [110]:
import re
from collections import Counter

# Join all of the text and force it to lowercase
all_text = " ".join(df['Text'].dropna()).lower()

# Use regex to find only words
words = re.findall(r'\b[a-z]+\b', all_text)

In [111]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Filter the list of words
clean_words = [w for w in words if w not in stop_words and len(w) > 2]
transcript_counts = Counter(clean_words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [112]:
# Sum the total number of words
total_cheers_words = sum(transcript_counts.values())

In [113]:
# Read in the SUBTLEXus corpus. Taking care to notice that the delimiter used for this file is "\t".
subtlex = pd.read_csv('/content/drive/MyDrive/CS215-Spring-AUGUSTUS/Projects/Final_Project/Data/subtlex_us-short.csv', sep="\t")
# Check it for red-flags
subtlex.head()

,Word,FREQcount,CDcount,FREQlow,Cdlow,SUBTLWF,Lg10WF,SUBTLCD,Lg10CD
0,the,1501908,8388,1339811,8388,29449.18,6.1766,100.00,3.9237
1,to,1156570,8383,1138435,8380,22677.84,6.0632,99.94,3.9235
2,a,1041179,8382,976941,8380,20415.27,6.0175,99.93,3.9234
3,you,2134713,8381,1595028,8376,41857.12,6.3293,99.92,3.9233
4,and,682780,8379,515365,8374,13387.84,5.8343,99.89,3.9232


In [114]:
# Put the SUBTLEXus data into a dict object
subtlex_dict = dict(zip(subtlex['Word'].str.lower(), subtlex['SUBTLWF']))
total_subtlex_words = 51000000

In [115]:
# Define a function that calculates the log likelihood when given the word count and the total number of words in dataset "a" and dataset "b"
def calculate_log_likelihood(count_a, total_a, count_b, total_b):
    # Expected frequencies
    e1 = total_a * (count_a + count_b) / (total_a + total_b)
    e2 = total_b * (count_a + count_b) / (total_a + total_b)
    # LL calculation
    t1 = count_a * np.log(count_a / e1 if count_a > 0 else 0.00001)
    t2 = count_b * np.log(count_b / e2 if count_b > 0 else 0.00001)

    return 2 * (t1 + t2)

In [116]:
results = []
for word, count in transcript_counts.items():
    if word in subtlex_dict:
        # Estimate the SUBTLEX count using (FreqPerMillion * TotalSize / 1,000,000)
        sub_count = (subtlex_dict[word] * total_subtlex_words) / 1_000_000
        # Use the function that we defined to calculate the LL score
        ll_score = calculate_log_likelihood(count, total_cheers_words, sub_count, total_subtlex_words)

        # Only keep words that are more frequent in Cheers than SUBTLEXus. We don't care about words that are uniquely absent in Cheers
        if (count / total_cheers_words) > (sub_count / total_subtlex_words):
            results.append({'word': word, 'll_score': ll_score, 'count': count})

In [117]:
# Sort the dataframe and print some of the ones with the highest LL score.
keywords_df = pd.DataFrame(results).sort_values(by='ll_score', ascending=False)
keywords_df.reset_index(drop=True, inplace=True)
print(keywords_df.head(5))

    word      ll_score  count
0    sam  27773.176734   4217
1  woody  16718.804377   1911
2   norm  10618.209878   1173
3   well   9528.710850   5676
4  diane   8403.556821   1093


Even just looking briefly we can see that most of the top words are the names of the main characters. This makes sense because a character name is something that is relatively unique to the show.
I'll quickly use my Java code to remove the names of the main characters from the list.

In [118]:
df_clean_keywords = pd.read_csv('/content/drive/MyDrive/CS215-Spring-AUGUSTUS/Projects/Final_Project/Big_Data/Cheers_Clean_Keywords.csv', sep=';', names=["word"])
print(df_clean_keywords.head())

    word
0   well
1   yeah
2   know
3    hey
4  right


We finally have a list of "Cheers keywords." These words appear uniquely often in the show.

## Character-Keyword Correlation

While the Cheers Keywords list is more meaningful, it still does not give us much insight into what defines each individual character. What if, for each of the top keywords, we find its correlation with each of the top characters, and then rank the words by how strong their correlation for any single character is?

In [119]:
from scipy.stats import pearsonr

# Initialize a list to hold all of our results
correlation_results = []

In [120]:
df_keywords_index = pd.read_csv('/content/drive/MyDrive/CS215-Spring-AUGUSTUS/Projects/Final_Project/Big_Data/Cheers_Keyword_Index.csv', sep=';')

In [121]:
# Iterate over every word in the keywords dataframe
for word in df_keywords_index.columns:

    # Iterate over every character in the names dataframe
    for character in df_names.columns:

        # Combine the two columns into a temporary dataframe and drop any NaNs just in case
        valid_data = pd.concat([df_keywords_index[word], df_names[character]], axis=1).dropna()

        # Extract the cleaned columns
        word_data = valid_data[word]
        char_data = valid_data[character]

        # Use pearsonr to calculate the correlation and p-value
        corr, p_val = pearsonr(word_data, char_data)

        # Add the results as a dictionary
        correlation_results.append({
            'Word': word,
            'Character': character,
            'Correlation': corr,
            'P_value': p_val
        })

In [122]:
# Convert the list of dictionaries into a DataFrame
df_all_correlations = pd.DataFrame(correlation_results)

# Drop any pairs that resulted in NaN
df_all_correlations = df_all_correlations.dropna()

# Sort by strongest positive correlation
df_all_correlations = df_all_correlations.sort_values(by='Correlation', ascending=False)
df_all_correlations.reset_index(drop=True, inplace=True)

In [123]:
df_all_correlations.head()

,Word,Character,Correlation,P_value
0,miss,Rebecca,0.124324,0.000000e+00
1,tavern,Gary,0.107769,1.020300e-285
2,afternoon,Norm,0.058100,4.011009e-84
3,beer,Norm,0.056659,4.342002e-80
4,olde,Gary,0.041680,3.848727e-44


Wow! Each of these word-character pairs can be explained by memorable character interactions in the show. While the correlations are weak, the p-values are extremely small. This means that the correlation value we found almost certainly indicates a real correlation in the data.

Let's organize this dataframe a bit more and clean it up a bit and then we can view it.

In [124]:
df_all_correlations = df_all_correlations.sort_values(by=['Word', 'Character'], ascending=[True, True])
df_all_correlations.reset_index(drop=True, inplace=True)

In [125]:
significant_words = df_all_correlations[df_all_correlations['P_value'] < 0.05]['Word'].unique()
df_filtered_correlations = df_all_correlations[df_all_correlations['Word'].isin(significant_words)]
df_filtered_correlations.reset_index(drop=True, inplace=True)

I removed words that didn't have a significant p-value with any of the characters.

I then exported the data so that I could implement it as an [interactive plot on my GitHub page](https://lafortua.github.io/Final_Project_CS-215/index.html#:~:text=Lots%20of%20Plots) for this project. Readers can choose which word the graph shows the data for.

In [126]:
df_filtered_correlations.to_csv('/content/drive/MyDrive/CS215-Spring-AUGUSTUS/Projects/Final_Project/Big_Data/Cheers_Character-Keyword_Correlation.csv', index=False, sep=";")

In [127]:
import plotly.express as px
# The target word chooses which word is displayed by the graph.
target_word = 'beer'
word_corrs = df_all_correlations[df_all_correlations['Word'] == target_word].sort_values(by='Character')

fig = px.bar(
    word_corrs,
    x='Character',
    y='Correlation',
    title=f'Correlation between "{target_word}" and Characters',
    color='Correlation',
    color_continuous_scale=px.colors.diverging.RdBu_r,
    color_continuous_midpoint=0,
    hover_data=['P_value']
)
fig.add_hline(y=0, line_width=1, line_color="black")
fig.update_layout(plot_bgcolor='white')
fig.show()

##Conclusion

The answer to the original question is yes. The data shows that the characters are clearly correlated with their job titles, favorite things, and common phrases.

I was a bit skeptical about being able to get meaningful data from the transcript without the speaker names, but it worked rather well. One part that was particularly challenging was working with such a large dataset that I myself created. Since the data was collected and organized by me, there were a lot of errors to fix. It then usually took me a long time to figure out what the errors were. This is because the large size of the dataset made it difficult to spot if it contained an error just by looking at it because I can't physically look at all of it.